# ToolUse Agent using Anthropic models

### State Diagram (Agent View)
```mermaid
stateDiagram-v2
direction TB

    INIT --> HAS_MESSAGE
    HAS_MESSAGE --> CHAT: condition_true
    HAS_MESSAGE --> TOOL_USE: condition_false

    CHAT--> IS_TOOL_CALL
    
    IS_TOOL_CALL --> FINAL: condition_false
    IS_TOOL_CALL --> TOOL_USE: condition_true

    TOOL_USE --> FINAL


```

### State Diagram (User View)
```mermaid
stateDiagram-v2
direction TB
    state "Start Agent" as StartAgent
    state "Continue Agent" as ContinueAgent
    state "Interrupt Agent" as InterruptAgent

    Initialize --> StartAgent : start_async
    StartAgent --> ContinueAgent : continue_async
    ContinueAgent --> InterruptAgent : interrupt_async
    InterruptAgent --> ContinueAgent : continue_async
    ContinueAgent --> ContinueAgent : continue_async
    ContinueAgent --> Terminate
    Terminate --> StartAgent : start_async
    Terminate --> InterruptAgent : interrupt_async

```



### a) Create Agent

In [1]:
from gai.asm.agents import ToolUseAgent
from gai.mcp.client.mcp_client import McpAggregatedClient
from gai.lib.config import config_helper

from gai.lib.tests import make_local_tmp
import os
here = make_local_tmp()
file_path = os.path.join(here, "monologue.json")
from gai.messages import FileMonologue
monologue = FileMonologue(agent_name="ToolUseAgent",file_path=file_path)

aggregated_client = McpAggregatedClient(["mcp-pseudo","mcp-time", "mcp-web"])
tools = await aggregated_client.list_tools()

# Create an artificial dialogue history for testing
from gai.messages import FileDialogue, MessagePydantic
messages = [
    MessagePydantic(**{
        'id': 'b1e5f98c-f6eb-47de-a6e2-387510d970f9', 
        'header': {
            'sender': 'User',
            'recipient': 'Sara',
            "timestamp": 1751308157.270983,
            "order": 0            
        }, 'body': {
            'type': 'chat.send',
            'dialogue_id': '00000000-0000-0000-0000-000000000000',
            'round_no': 0,
            'step_no': 0,
            'role': "user",
            'content': 'It is a very nice weather in Singapore right now.',
        }
    }),
    MessagePydantic(**{
        'id': 'abbc7961-45dc-4973-aaf4-a6224ed35d37', 
        'header': {
            'sender': 'Sara',
            'recipient': 'User',
            "timestamp": 1751308167.3488164,
            "order": 1
        }, 'body': {
            'type': 'chat.reply',
            'dialogue_id': '00000000-0000-0000-0000-000000000000',
            'round_no': 0, 
            'step_no': 1,
            'chunk_no':10,
            'chunk':'<eom>',
            'role': "assistant",
            'content': 'Yes, it is! The weather in Singapore is typically warm and humid, with occasional rain showers. It\'s a great time to enjoy outdoor activities or relax indoors with a cool drink. How can I assist you today?'
        }
    })]
from gai.lib.constants import DEFAULT_GUID
file_path = os.path.join(here, f"{DEFAULT_GUID}.json")
dialogue = FileDialogue(messages=messages,file_path=file_path)
recap = dialogue.extract_recap()

agent = ToolUseAgent(
    agent_name="ToolUseAgent",
    llm_config=config_helper.get_client_config(
        {
            "client_type": "anthropic",
            "model": "claude-sonnet-4-20250514",
            "extra": {
                "max_tokens": 32000,
                "temperature": 0.7,
                "top_p": 0.95,
                "tools": True,
                "stream": True,
            },
        }
    ),
    aggregated_client=aggregated_client,
    monologue=monologue,
    recap=recap
)

### reset monologue (optional)

In [2]:
monologue.reset()


### b) start

The location of the public holiday is inferred from the dialogue context.

In [3]:
goal = "When is the next public holiday?"
user_message=f"""
## Goal:
{goal}
## Instructions:
- You have the following tools at your disposal: {tools}
"""

resp=agent.start(user_message=user_message)
# Stream the response
async for chunk in resp:
    if chunk:
        if isinstance(chunk, str):
            print(chunk, end="", flush=True)

I'll help you find the next public holiday in Singapore. Let me search for the current public holiday information.Now let me get the current date to determine which is the next upcoming public holiday:

### c) Show monologue

In [4]:
import json

# Show the monologue
print("\n───────────────────────── MONOLOGUE START ─────────────────────────")
messages = agent.fsm.monologue.list_messages()
for message in messages[-2:]:
    print(json.dumps(message.model_dump(), indent=4))
print("───────────────────────── MONOLOGUE END ─────────────────────────\n")

# Print memory size
mem_size = agent.fsm.monologue.get_total_size()
print("Total char size=", mem_size)


───────────────────────── MONOLOGUE START ─────────────────────────
{
    "id": "23018d1b-9ed0-40a5-84c0-8c9c8fc0fc41",
    "header": {
        "sender": "User",
        "recipient": "ToolUseAgent",
        "timestamp": 1752631794.085841,
        "order": 2
    },
    "body": {
        "type": "monologue",
        "state_name": "TOOL_USE",
        "step_no": 4,
        "content_type": "text",
        "role": "user",
        "content": [
            {
                "type": "tool_result",
                "tool_use_id": "toolu_01MFt7x4DvRT7oSLqHDbtP5i",
                "content": "{\n  \"query\": \"Singapore public holidays 2024 2025 next upcoming\",\n  \"chunks\": [\n    {\n      \"index\": 0,\n      \"link_title\": \"https://www.mom.gov.sg/employment-practices/public-holidays\",\n      \"link\": \"https://www.mom.gov.sg/employment-practices/public-holidays\",\n      \"chunk\": \"The next public holiday is 9 August, Saturday National Day 2024 2025 2026 Date Day Holiday image Holiday n

### d) resume

In [5]:
resp = agent.resume()
# Stream the response
async for chunk in resp:
    if chunk:
        if isinstance(chunk, str):
            print(chunk, end="", flush=True)


Based on the current date (July 16, 2025) and the official public holiday information from Singapore's Ministry of Manpower, the **next public holiday in Singapore is National Day on Saturday, August 9, 2025**.

Here's a quick overview of the upcoming public holidays in Singapore for the rest of 2025:

1. **National Day** - August 9, 2025 (Saturday)
2. **Deepavali** - October 20, 2025 (Monday)
3. **Christmas Day** - December 25, 2025 (Thursday)

National Day is Singapore's most significant national celebration, commemorating the country's independence. It's always celebrated on August 9th each year with various festivities, parades, and fireworks displays across the island.

### e) User interrupt agent

Should be able to interrupt the agent and continue with original task.

In [6]:
resp = agent.interrupt(user_message="Tell me a one paragraph joke.")
# Stream the response
async for chunk in resp:
    if chunk:
        if isinstance(chunk, str):
            print(chunk, end="", flush=True)


Here's a quick joke for you:

A man walks into a library and asks for books on paranoia. The librarian whispers, "They're right behind you!" The man spins around frantically, only to see the librarian pointing to the shelf directly behind him. He sheepishly grabs a book, but as he's checking out, the librarian leans in and whispers, "You know, everyone who reads these books thinks someone is following them." The man nervously laughs and says, "That's ridiculous!" Then he walks out... and immediately starts looking over his shoulder every few steps.

Now, getting back to our earlier conversation about Singapore's weather and public holidays - is there anything else you'd like to know about upcoming holidays or perhaps some suggestions for activities to enjoy during the nice weather you mentioned?

### f) Show dialogue

In [8]:
dialogue.add_user_message(recipient='Sara', content=goal)
dialogue.add_assistant_message(sender='Sara', chunk="<eom>", content=agent.final_output())
for msg in dialogue.list_messages():
    print(f"{msg.header.sender}: {msg.body.content}")

User: It is a very nice weather in Singapore right now.
Sara: Yes, it is! The weather in Singapore is typically warm and humid, with occasional rain showers. It's a great time to enjoy outdoor activities or relax indoors with a cool drink. How can I assist you today?
User: Sara, When is the next public holiday?
Sara: Here's a little joke for you:

A man walks into a library and asks for books on paranoia. The librarian whispers, "They're right behind you!" The man spins around frantically, only to see the librarian pointing to the shelf directly behind him. Feeling embarrassed, he grabs a book and sits down to read. After a few minutes, he notices everyone in the library is staring at him. He gets increasingly nervous and finally approaches the librarian again, asking, "Why is everyone looking at me?" The librarian replies, "Well, you're the only one who checked out 'How to Deal with Paranoia' and then proceeded to read it while constantly looking over your shoulder and whispering 'the

---
## Scenario 1 - Agent interrupts itself and User has no input

- LLM Interrupts itself to ask user question
- User cannot continue because LLM is waiting for user input

In [7]:
import os
from gai.asm.agents import ToolUseAgent
from gai.mcp.client.mcp_client import McpAggregatedClient
from gai.lib.config import config_helper
from gai.lib.tests import make_local_tmp
from gai.messages import FileMonologue

here = make_local_tmp()
file_path = os.path.join(here, "monologue.json")
monologue = FileMonologue(agent_name="ToolUseAgent",file_path=file_path)
aggregated_client = McpAggregatedClient(["mcp-pseudo","mcp-time", "mcp-web"])
tools = await aggregated_client.list_tools()
agent = ToolUseAgent(
    agent_name="ToolUseAgent",
    llm_config=config_helper.get_client_config(
        {
            "client_type": "anthropic",
            "model": "claude-sonnet-4-20250514",
            "extra": {
                "max_tokens": 32000,
                "temperature": 0.7,
                "top_p": 0.95,
                "tools": True,
                "stream": True,
            },
        }
    ),
    monologue=monologue,
    aggregated_client=aggregated_client,
)

In [8]:
monologue.reset()

goal = "What time is it right now?"
user_message = f"""
## Goal:
{goal}
## Instructions:
- You have the following tools at your disposal: {tools}
- Use "user_input" to ask for timezone before you start.
"""

resp = agent.start(user_message=user_message)
# Stream the response
async for chunk in resp:
    if chunk:
        if isinstance(chunk, str):
            print(chunk, end="", flush=True)


I'll help you get the current time! First, let me ask you about your timezone preference.What timezone would you like me to use for the current time? Please provide the timezone in IANA format (e.g., America/New_York, Europe/London, Asia/Tokyo, etc.). If you're not sure, just let me know your location or preferred timezone name.

Confirm that "resume" returns nothing because the agent is waiting for user input.

In [9]:
resp = agent.resume()
# Stream the response
can_resume = False
async for chunk in resp:
    can_resume = True
if not can_resume:
    print("\nAgent cannot resume because User did not provide input.")
assert not can_resume, "Agent should not be able to resume while waiting for input."



Agent cannot resume because User did not provide input.


---
## Scenario 2: LLM Interrupts itself and User has input

- LLM Interrupts itself to ask user question
- User responds with answer

In [10]:
import os
from gai.asm.agents import ToolUseAgent
from gai.mcp.client.mcp_client import McpAggregatedClient
from gai.lib.config import config_helper
from gai.lib.tests import make_local_tmp
from gai.messages import FileMonologue

here = make_local_tmp()
file_path = os.path.join(here, "monologue.json")
monologue = FileMonologue(agent_name="ToolUseAgent",file_path=file_path)
aggregated_client = McpAggregatedClient(["mcp-pseudo","mcp-time", "mcp-web"])
tools = await aggregated_client.list_tools()
agent = ToolUseAgent(
    agent_name="ToolUseAgent",
    llm_config=config_helper.get_client_config(
        {
            "client_type": "anthropic",
            "model": "claude-sonnet-4-20250514",
            "extra": {
                "max_tokens": 32000,
                "temperature": 0.7,
                "top_p": 0.95,
                "tools": True,
                "stream": True,
            },
        }
    ),
    monologue=monologue,
    aggregated_client=aggregated_client,
)

In [11]:
monologue.reset()

goal = "What time is it right now?"
user_message = f"""
## Goal:
{goal}
## Instructions:
- You have the following tools at your disposal: {tools}
- Use "user_input" to ask for timezone before you start.
"""

resp = agent.start(user_message=user_message)
# Stream the response
async for chunk in resp:
    if chunk:
        if isinstance(chunk, str):
            print(chunk, end="", flush=True)


I'll help you get the current time. First, let me ask for your timezone preference.I need to know your timezone to provide you with the accurate current time. Could you please provide your timezone? You can specify it as:
- A timezone name (e.g., "America/New_York", "Europe/London", "Asia/Shanghai", "America/Los_Angeles")
- Or let me know your city/region and I can help determine the timezone

What timezone would you like me to use for the current time?

Confirm user cannot continue because LLM is waiting for user input.

In [12]:
resp = agent.resume()
# Stream the response
can_resume = False
async for chunk in resp:
    can_resume = True
if not can_resume:
    print("\nAgent cannot resume because User did not provide input.")
assert not can_resume, "Agent should not be able to resume while waiting for input."



Agent cannot resume because User did not provide input.


Confirm that "resume" with message will work when user responds.

In [13]:
resp = agent.resume(user_message="Use SGT")
# Stream the response
can_resume = False
async for chunk in resp:
    can_resume = True
    if chunk:
        if isinstance(chunk, str):
            print(chunk, end="", flush=True)
assert can_resume, "Agent should be able to resume after user provides input."


The current time in Singapore (SGT) is:

**2025-07-16 10:12:32**

This is Tuesday, July 16th, 2025 at 10:12:32 AM Singapore time.

User can resume normally.

In [14]:
resp = agent.resume()
async for chunk in resp:
    if chunk and isinstance(chunk, str):
        print(chunk, end="", flush=True)